## MMR

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain


In [4]:
from dotenv import load_dotenv
# Load environment variables
load_dotenv()
import os

# Initialize LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="qwen/qwen3-32b"
)

# Invoke model
response = llm.invoke("Hi")
print(response)

content='<think>\nOkay, the user said "Hi". I need to respond appropriately. Let me start by greeting them back. Maybe something friendly and open-ended to encourage conversation. I should ask how they\'re doing and offer help if they need anything. Keep it simple and welcoming. Let me put that together.\n</think>\n\nHello! How can I assist you today? 😊' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 9, 'total_tokens': 84, 'completion_time': 0.131735374, 'prompt_time': 0.000354605, 'queue_time': 0.047636512, 'total_time': 0.132089979}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--019ea853-7a8e-7da3-9fd1-70fbd42a2032-0' usage_metadata={'input_tokens': 9, 'output_tokens': 75, 'total_tokens': 84}


In [5]:
#load and create a chunks
##load

loader = TextLoader("sample_dataset.txt")
raw_docs = loader.load()

##textsplitter (split text into docs chunks)
text_splitter = RecursiveCharacterTextSplitter (chunk_size = 300, chunk_overlap = 50)
docs = text_splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'sample_dataset.txt'}, page_content='BMW is a German luxury automobile manufacturer founded in 1916.\nThe company is headquartered in Munich, Germany.\nBMW produces premium cars, motorcycles, and electric vehicles.\nThe BMW i4 and BMW iX are popular electric vehicle models.\nGroq develops AI inference hardware for large language models.'),
 Document(metadata={'source': 'sample_dataset.txt'}, page_content='Its LPU architecture is designed for low-latency AI inference.\nGroq focuses on accelerating generative AI applications.\nRetrieval-Augmented Generation (RAG) combines retrieval and generation.\nRAG enables language models to use external knowledge sources.'),
 Document(metadata={'source': 'sample_dataset.txt'}, page_content='This improves answer accuracy and reduces hallucinations.\nChromaDB is a vector database used in AI applications.\nIt stores embeddings and supports semantic search.\nVector databases are commonly used in RAG systems.\nEmbeddings con

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
#vectorstore embedding
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13931.87it/s]


In [12]:
#FAISS Vectore store

vectoretore = FAISS.from_documents(docs,embedding)
retriever = vectoretore.as_retriever(search_type="mmr",search_kwargs= {"k":3})

In [10]:
prompt = PromptTemplate.from_template("""
Answer the question based on the context.
Context:
{context}                                     
Question: {input}
                                      """)

In [11]:
#RAG Pipeline

document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)

In [15]:
rag_chain = create_retrieval_chain(retriever=retriever,combine_docs_chain=document_chain)

In [16]:
query = {"input":"what is BMW?"}
res = rag_chain.invoke(query)
print(res['answer'])

<think>
Okay, so I need to answer the question "what is BMW?" based on the given context. Let me start by reading through the context carefully.

The context says that BMW is a German luxury automobile manufacturer founded in 1916. The headquarters are in Munich, Germany. They make premium cars, motorcycles, and electric vehicles. Specific models mentioned are the BMW i4 and BMW iX as popular electric vehicles. There's also a mention of Groq developing AI inference hardware for large language models, but that seems unrelated to BMW. Then there's information about semantic chunking and re-ranking, which are about text processing and retrieval systems, and LangChain as a framework for LLM applications. However, those parts don't relate to the question about BMW directly.

So focusing on the relevant parts, BMW is a car company from Germany, established in 1916. They produce luxury vehicles, including electric models. The key points are: German, founded 1916, Munich headquarters, premium 